In [2]:
# # A partir de votre notebook lister les ficheirs dans les conteneus de données azureraw et clean


In [1]:
import os
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# Charge les variables du fichier .env (si exécuté hors Docker en local)
load_dotenv()

# Récupération des identifiants depuis l'environnement
account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")

# Vérification rapide pour éviter une erreur floue si la variable est absente
if not account_name or not account_key:
  raise ValueError(
      "Les variables d'environnement Azure ne sont pas définies."
  )

# Initialisation du client Blob Service
blob_service_client = BlobServiceClient(
    account_url=f"https://{account_name}.blob.core.windows.net",
    credential=account_key,
)

print("Exploration des conteneurs et de leur contenu :\n")

# Lister tous les conteneurs du compte
containers = blob_service_client.list_containers()
for container in containers:
  container_name = container.name
  print(f"📁 Conteneur : {container_name}")

  # Créer un client pour ce conteneur spécifique
  container_client = blob_service_client.get_container_client(container_name)

  # Lister les blobs (fichiers) à l'intérieur du conteneur
  blobs = list(container_client.list_blobs())

  if len(blobs) > 0:
    for blob in blobs:
      print(f"    └── 📄 {blob.name} (Taille : {blob.size} octets)")
  else:
    print("    └── (Conteneur vide)")

  print("-" * 40)

Exploration des conteneurs et de leur contenu :

📁 Conteneur : clean
    └── (Conteneur vide)
----------------------------------------
📁 Conteneur : raw
    └── 📄 categories.csv (Taille : 428 octets)
    └── 📄 customers.csv (Taille : 11999 octets)
    └── 📄 employees.csv (Taille : 4140 octets)
    └── 📄 order_details.csv (Taille : 47311 octets)
    └── 📄 orders.csv (Taille : 101130 octets)
    └── 📄 products.csv (Taille : 4677 octets)
    └── 📄 shippers.csv (Taille : 224 octets)
    └── 📄 suppliers.csv (Taille : 4150 octets)
----------------------------------------


In [2]:
import os

print("AZURE_STORAGE_ACCOUNT_NAME:", os.getenv("AZURE_STORAGE_ACCOUNT_NAME"))
print(
    "AZURE_STORAGE_ACCOUNT_KEY:",
    "Présente (masquée)"
    if os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
    else "ABSENTE",
)
print("POSTGRES_DB:", os.getenv("POSTGRES_DB"))

AZURE_STORAGE_ACCOUNT_NAME: gregelbadjouristorage
AZURE_STORAGE_ACCOUNT_KEY: Présente (masquée)
POSTGRES_DB: tradecorp


In [11]:
# Essaie envoir fichiers raw
import os
from pathlib import Path
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# Chargement des identifiant Azure depuis le fichier .env
load_dotenv()
account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")

if not account_name or not account_key:
    raise ValueError("Les variables d'environnement Azure ne sont pas définies mon salo.")

# Init du client Azure Blob Storage
blob_service_client = BlobServiceClient(
    account_url=f"https://{account_name}.blob.core.windows.net",
    credential=account_key,
)

container_name = "raw"
container_client = blob_service_client.get_container_client(container_name)

# Ciblage répertoire monté dans Docker, transfert avec vérification
local_raw_dir = Path("/home/jovyan/data/raw")
files = list(local_raw_dir.glob("*.csv"))

print(f"Début de l'analyse de {len(files)} fichier(s) vers Azure ({container_name})\n")

for file_path in files:
    blob_name = file_path.name
    blob_client = container_client.get_blob_client(blob_name)
    
    # Vérification de l'existence du fichier dans le Data Lake
    if blob_client.exists():
        print(f" Attention : Le fichier '{blob_name}' existe déjà dans Azure. Il va être écrasé.")
    
    print(f" Upload réussi : {file_path.name} -> {container_name}/{blob_name}")
    with open(file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)

print("\nTransfert terminé avec succès ! Assieds toi et croissantes quelqu'un ")


Début de l'analyse de 8 fichier(s) vers Azure (raw)

 Attention : Le fichier 'categories.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : categories.csv -> raw/categories.csv
 Attention : Le fichier 'shippers.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : shippers.csv -> raw/shippers.csv
 Attention : Le fichier 'order_details.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : order_details.csv -> raw/order_details.csv
 Attention : Le fichier 'employees.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : employees.csv -> raw/employees.csv
 Attention : Le fichier 'products.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : products.csv -> raw/products.csv
 Attention : Le fichier 'suppliers.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : suppliers.csv -> raw/suppliers.csv
 Attention : Le fichier 'customers.csv' existe déjà dans Azure. Il va être écrasé.
 Upload réussi : customers.csv -> raw/custome

In [3]:
import os
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, initcap, lit, lower, round, trim, upper, when
from pyspark.sql.types import DateType, DoubleType, IntegerType

load_dotenv()

# --- CONNEXION ET GESTION ADLS GEN2 ---

def get_blob_service_client() -> BlobServiceClient:
    """Initialise et retourne le client Blob Service pour Azure ADLS Gen2."""
    account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
    account_key = os.getenv("AZURE_STORAGE_ACCOUNT_KEY")
    connect_str = f"DefaultEndpointsProtocol=https;AccountName={account_name};AccountKey={account_key};EndpointSuffix=core.windows.net"
    return BlobServiceClient.from_connection_string(connect_str)


def download_blob_to_local(blob_service_client: BlobServiceClient, container_name: str, blob_name: str, download_path: str):
    """Télécharge un fichier spécifique depuis un conteneur ADLS Gen2 vers le système local."""
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    os.makedirs(os.path.dirname(download_path), exist_ok=True)
    with open(download_path, "wb") as download_file:
        download_file.write(blob_client.download_blob().readall())


# --- FONCTIONS DE NETTOYAGE PAR TABLE ---

def clean_customers(df: DataFrame) -> DataFrame:
    """Nettoie la table des clients : trim, normalisation et déduplication."""
    return (
        df.withColumn("company_name", trim(col("company_name")))
          .withColumn("contact_name", initcap(trim(col("contact_name"))))
          .withColumn("country", upper(trim(col("country"))))
          .dropDuplicates(["customer_id"])
    )


def clean_orders(df: DataFrame) -> DataFrame:
    """Nettoie la table des commandes : filtrage des non-livrées, retypage et renommage."""
    return (
        df.filter(col("shipped_date").isNotNull())
          .withColumn("order_date", col("order_date").cast(DateType()))
          .withColumn("required_date", col("required_date").cast(DateType()))
          .withColumn("shipped_date", col("shipped_date").cast(DateType()))
          .withColumn("freight", col("freight").cast(DoubleType()))
          .withColumnRenamed("ship_via", "shipper_id")
          .withColumn("is_shipped", when(col("shipped_date").isNotNull(), True).otherwise(False))
    )


def clean_order_details(df: DataFrame) -> DataFrame:
    """Nettoie le détail des lignes de commande et renomme les colonnes en français."""
    return (
        df.withColumn("unit_price", col("unit_price").cast(DoubleType()))
          .withColumn("quantity", col("quantity").cast(IntegerType()))
          .withColumn("discount", col("discount").cast(DoubleType()))
          .withColumnRenamed("unit_price", "prix_unitaire")
          .withColumnRenamed("quantity", "quantite")
    )


def add_sous_total(df: DataFrame) -> DataFrame:
    """Calcule le sous-total d'une ligne de commande (prix × quantité × (1 - remise))."""
    return df.withColumn(
        "sous_total",
        round(col("prix_unitaire") * col("quantite") * (lit(1) - col("discount")), 2)
    )


def clean_employees(df: DataFrame) -> DataFrame:
    """Filtre les colonnes utiles des employés et crée full_name."""
    cols_to_keep = ["employee_id", "first_name", "last_name", "title", "hire_date", "city", "country"]
    return (
        df.select([c for c in cols_to_keep if c in df.columns])
          .withColumn("full_name", trim(col("first_name")) + lit(" ") + trim(col("last_name")))
    )


def clean_products(df: DataFrame) -> DataFrame:
    """Nettoie la table des produits, convertit le prix et ajoute l'indicateur de stock."""
    return (
        df.withColumn("unit_price", col("unit_price").cast(DoubleType()))
          .withColumn("en_stock", when(col("units_in_stock") > 0, True).otherwise(False))
    )

In [4]:
print(os.getenv("AZURE_STORAGE_ACCOUNT_NAME"))

gregelbadjouristorage
